# 03 — Previsão de volume D+1

**Objetivo:** prever o número de incidentes elegíveis a KPI no dia seguinte, por entidade (global, produto, equipe, categoria, prioridade e ativo).

**Algoritmo vencedor no último treino da plataforma:** XGBoost (XGBRegressor) · **métrica principal:** MAE (e MASE contra a baseline sazonal)

Este notebook contém o treino completo do modelo, célula por célula: divisão temporal, candidatos,
competição na validação, reajuste, avaliação no teste e explicabilidade. O nome do arquivo reflete
o vencedor registrado na plataforma, mas **a escolha não está fixada no código**: todos os
candidatos competem de novo a cada execução, e vence quem tiver a melhor métrica na validação.

> Pré-requisito: executar antes o `02_engenharia_de_features.ipynb`, que gera `dados/base.pkl`.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

base = pd.read_pickle(DADOS / 'base.pkl')
print('Base carregada:', f"{len(base['incidentes']):,} incidentes |",
      f"regime a partir de {base['inicio_regime']:%d/%m/%Y}")

## 1. Recorte do problema e divisão temporal

In [ ]:
DIAS_TESTE = 28
DIAS_VALIDACAO = 28

COLUNAS_IDENTIFICACAO = ['dimensao', 'chave', 'data_origem', 'data_alvo']
COLUNAS_ALVO = ['alvo', 'alvo_violacoes', 'alvo_criticos', 'alvo_total']


def dividir(matriz, dias_teste=DIAS_TESTE, dias_validacao=DIAS_VALIDACAO, coluna='data_alvo'):
    """Treino / validação / teste cortados pela DATA DO ALVO, não pela data de origem.

    Uma linha de origem 20/11 com horizonte 7 enxerga o dia 27/11; cortar pela origem deixaria
    essa linha no treino com o dia 27 já dentro da validação. Cortar pelo alvo elimina a
    sobreposição em qualquer horizonte.
    """
    fim = pd.Timestamp(matriz[coluna].max())
    inicio_teste = fim - pd.Timedelta(days=dias_teste - 1)
    inicio_validacao = inicio_teste - pd.Timedelta(days=dias_validacao)
    alvo = matriz[coluna]
    return alvo < inicio_validacao, (alvo >= inicio_validacao) & (alvo < inicio_teste), alvo >= inicio_teste


def descrever_particoes(matriz, mascaras, coluna='data_alvo'):
    linhas = {}
    for nome, mascara in zip(('treino', 'validação', 'teste'), mascaras):
        recorte = matriz.loc[mascara, coluna]
        linhas[nome] = {'linhas': int(mascara.sum()), 'início': recorte.min().date(), 'fim': recorte.max().date()}
    return pd.DataFrame(linhas).T


def colunas_de_features(matriz):
    """Tudo que não é identificação nem alvo é feature."""
    excluir = set(COLUNAS_IDENTIFICACAO + COLUNAS_ALVO)
    return [c for c in matriz.columns if c not in excluir and matriz[c].dtype != object]

In [ ]:
HORIZONTES = [1]
matriz = base['matriz'][base['matriz']['horizonte'].isin(HORIZONTES)].copy().reset_index(drop=True)
treino, validacao, teste = dividir(matriz)

colunas = colunas_de_features(matriz)
X = matriz[colunas].astype(float).fillna(0.0).replace([np.inf, -np.inf], 0.0)
if len(HORIZONTES) == 1:
    X = X.drop(columns=['horizonte'])   # com horizonte único a coluna vira constante
y = matriz['alvo'].astype(float)

X_treino, X_validacao, X_teste = X[treino], X[validacao], X[teste]
y_treino, y_validacao, y_teste = y[treino], y[validacao], y[teste]

print(f'Features: {X.shape[1]} | entidades: {matriz["chave"].nunique()}')
descrever_particoes(matriz, (treino, validacao, teste))

## 2. Métrica de avaliação

In [ ]:
from sklearn.metrics import r2_score


def metricas_regressao(y_real, y_previsto, y_baseline=None):
    """MAE, RMSE, MAPE (só dias com volume > 0), R², viés e — com baseline — MASE e ganho."""
    y = np.asarray(y_real, dtype=float)
    p = np.asarray(y_previsto, dtype=float)
    erro = y - p
    nao_zero = y > 0
    resultado = {
        'mae': float(np.mean(np.abs(erro))),
        'rmse': float(np.sqrt(np.mean(erro ** 2))),
        'mape': float(np.mean(np.abs(erro[nao_zero] / y[nao_zero])) * 100) if nao_zero.any() else None,
        'r2': float(r2_score(y, p)) if len(y) > 1 and np.std(y) else None,
        'vies': float(np.mean(erro)),
        'n': int(len(y)),
    }
    if y_baseline is not None:
        b = np.asarray(y_baseline, dtype=float)
        mae_baseline = float(np.mean(np.abs(y - b)))
        resultado['mae_baseline'] = mae_baseline
        resultado['mase'] = resultado['mae'] / mae_baseline if mae_baseline else None
        resultado['ganho_vs_baseline'] = (1 - resultado['mae'] / mae_baseline) * 100 if mae_baseline else None
    return resultado

## 3. Candidatos

Baselines entram como estimadores de verdade, na mesma partição e na mesma métrica — é o que
permite afirmar que o ML acrescenta algo sobre a regra simples.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


class PreverColuna(BaseEstimator, RegressorMixin):
    """Baseline que devolve uma coluna já presente nas features.
    Com `mesmo_dow_1` é a baseline sazonal ingênua (repetir o mesmo dia da semana anterior)."""

    def __init__(self, coluna='mesmo_dow_1'):
        self.coluna = coluna

    def fit(self, X, y=None):
        self.indice_ = list(X.columns).index(self.coluna)
        return self

    def predict(self, X):
        return np.asarray(X.iloc[:, self.indice_], dtype=float)


# Grade curta de propósito: com dezenas de milhares de linhas, busca exaustiva escolheria ruído da
# validação. Toda floresta tem max_depth limitado ANTES da competição (restrição de tamanho do
# artefato para deploy), para que o limite não vire desempate depois de ver o resultado.
CANDIDATOS = {
    'Baseline sazonal': {
        'tipo': 'baseline',
        'estimador': lambda: PreverColuna(coluna='mesmo_dow_1'),
        'grade': [{}],
    },
    'Baseline média 7d': {
        'tipo': 'baseline',
        'estimador': lambda: PreverColuna(coluna='media_7'),
        'grade': [{}],
    },
    'Ridge': {
        'tipo': 'linear',
        'estimador': lambda: Pipeline([('escala', StandardScaler()), ('modelo', Ridge(random_state=SEMENTE))]),
        'grade': [{'modelo__alpha': a} for a in (1.0, 10.0, 100.0)],
    },
    'Random Forest': {
        'tipo': 'arvore',
        'estimador': lambda: RandomForestRegressor(random_state=SEMENTE, n_jobs=-1),
        'grade': [
            {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': 10},
            {'n_estimators': 300, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_depth': 14},
            {'n_estimators': 300, 'min_samples_leaf': 10, 'max_features': 0.8, 'max_depth': 18},
        ],
    },
    'XGBoost': {
        'tipo': 'arvore',
        'estimador': lambda: XGBRegressor(random_state=SEMENTE, n_jobs=-1, tree_method='hist',
                                          objective='reg:squarederror'),
        'grade': [
            {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 5,
             'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0},
            {'n_estimators': 400, 'learning_rate': 0.08, 'max_depth': 4,
             'subsample': 0.9, 'colsample_bytree': 0.7, 'reg_lambda': 2.0},
            {'n_estimators': 900, 'learning_rate': 0.03, 'max_depth': 6,
             'subsample': 0.8, 'colsample_bytree': 0.6, 'reg_lambda': 3.0},
        ],
    },
    'LightGBM': {
        'tipo': 'arvore',
        'estimador': lambda: LGBMRegressor(random_state=SEMENTE, n_jobs=-1, verbose=-1),
        'grade': [
            {'n_estimators': 600, 'learning_rate': 0.05, 'num_leaves': 31,
             'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8},
            {'n_estimators': 400, 'learning_rate': 0.08, 'num_leaves': 15,
             'min_child_samples': 40, 'colsample_bytree': 0.7},
        ],
    },
}
for nome, definicao in CANDIDATOS.items():
    print(f'{nome:<18} {definicao["tipo"]:<9} {len(definicao["grade"])} configuração(ões)')

## 4. Competição: treino no treino, escolha na validação

In [ ]:
import time


def instanciar(definicao, parametros):
    estimador = definicao['estimador']()
    return estimador.set_params(**parametros) if parametros else estimador


def competir(candidatos, X_treino, y_treino, X_validacao, y_validacao, avaliar, chave_metrica, maior_melhor=False):
    """Protocolo de seleção:
    1. cada configuração de cada algoritmo treina SÓ no treino;
    2. o vencedor é escolhido pela VALIDAÇÃO (o teste não participa da decisão);
    3. depois o vencedor é reajustado em treino + validação e medido UMA vez no teste.
    """
    tabela, melhor = [], None
    for nome, definicao in candidatos.items():
        for parametros in definicao['grade']:
            modelo = instanciar(definicao, parametros)
            inicio = time.perf_counter()
            modelo.fit(X_treino, y_treino)
            segundos = time.perf_counter() - inicio
            resultado = avaliar(modelo, X_validacao, y_validacao)
            tabela.append({'candidato': nome, 'tipo': definicao['tipo'], 'hiperparametros': parametros,
                           'segundos_treino': round(segundos, 1), **resultado})
            pontuacao = resultado.get(chave_metrica)
            if pontuacao is None:
                continue
            if melhor is None or (pontuacao > melhor['pontuacao'] if maior_melhor else pontuacao < melhor['pontuacao']):
                melhor = {'nome': nome, 'definicao': definicao, 'parametros': parametros,
                          'pontuacao': pontuacao, 'modelo': modelo}
    tabela = pd.DataFrame(tabela)
    tabela['vencedor'] = (tabela['candidato'] == melhor['nome']) & (
        tabela['hiperparametros'].astype(str) == str(melhor['parametros']))
    return melhor, tabela


def reajustar(melhor, X, y):
    """Mesma configuração vencedora, reajustada sobre treino + validação."""
    modelo = instanciar(melhor['definicao'], melhor['parametros'])
    modelo.fit(X, y)
    return modelo

In [ ]:
def avaliar_regressao(modelo, X, y):
    previsto = np.clip(modelo.predict(X), 0, None)
    return metricas_regressao(y, previsto, X['mesmo_dow_1'].to_numpy())


melhor, tabela = competir(CANDIDATOS, X_treino, y_treino, X_validacao, y_validacao,
                          avaliar=avaliar_regressao, chave_metrica='mae')
print('Vencedor na validação:', melhor['nome'], melhor['parametros'])
tabela[['candidato', 'tipo', 'hiperparametros', 'mae', 'rmse', 'mase', 'r2', 'segundos_treino', 'vencedor']] \
    .sort_values('mae').round(3)

## 5. Reajuste em treino + validação e avaliação única no teste

In [ ]:
X_ajuste = pd.concat([X_treino, X_validacao])
y_ajuste = pd.concat([y_treino, y_validacao])
modelo = reajustar(melhor, X_ajuste, y_ajuste)

previsto_teste = np.clip(modelo.predict(X_teste), 0, None)
resultado_teste = metricas_regressao(y_teste, previsto_teste, X_teste['mesmo_dow_1'].to_numpy())
pd.Series(resultado_teste, name='teste (model_d1)').round(3)

In [ ]:
# O MAE do painel mistura escalas (global vs. produto pequeno): o recorte por dimensão evita leitura errada.
identificacao_teste = matriz.loc[teste].reset_index(drop=True)
por_dimensao = {}
for dimensao, indices in identificacao_teste.groupby('dimensao').groups.items():
    posicao = np.asarray(indices)
    por_dimensao[dimensao] = metricas_regressao(
        y_teste.to_numpy()[posicao], previsto_teste[posicao], X_teste['mesmo_dow_1'].to_numpy()[posicao])
pd.DataFrame(por_dimensao).T[['mae', 'mae_baseline', 'mase', 'ganho_vs_baseline', 'vies', 'n']].round(3)

In [ ]:
# Real × previsto na série global do teste.
mascara_global = (identificacao_teste['dimensao'] == 'global').to_numpy()
if len(HORIZONTES) > 1:
    mascara_global = mascara_global & (identificacao_teste['horizonte'] == 1).to_numpy()
datas = identificacao_teste.loc[mascara_global, 'data_alvo']
fig, eixo = plt.subplots()
eixo.plot(datas, y_teste.to_numpy()[mascara_global], marker='o', label='real', color='#0f172a')
eixo.plot(datas, previsto_teste[mascara_global], marker='o', label='previsto', color='#2563eb')
eixo.plot(datas, X_teste['mesmo_dow_1'].to_numpy()[mascara_global], ls='--', label='baseline sazonal', color='#94a3b8')
eixo.set_title('Série global no conjunto de teste')
eixo.legend(); plt.show()

## 6. Faixa da previsão

A margem é calibrada na **validação** como fração da escala da entidade (|erro| ÷ média de 7 dias,
percentil 80) e a cobertura é medida no **teste**.

In [ ]:
COBERTURA_ALVO = 0.80


def margens_relativas(residuos, escala, chaves, cobertura=COBERTURA_ALVO, minimo=30):
    relativo = np.abs(residuos) / np.maximum(escala, 1.0)
    geral = float(np.quantile(relativo, cobertura))
    por_chave = {}
    for valor in pd.unique(chaves):
        recorte = relativo[chaves == valor]
        if len(recorte) >= minimo:
            por_chave[str(valor)] = float(np.quantile(recorte, cobertura))
    return {'geral': geral, 'por_chave': por_chave}


previsto_validacao = np.clip(melhor['modelo'].predict(X_validacao), 0, None)
margens = margens_relativas(y_validacao.to_numpy() - previsto_validacao, X_validacao['media_7'].to_numpy(),
                            matriz.loc[validacao, 'dimensao'].to_numpy())
margem_teste = np.array([margens['por_chave'].get(d, margens['geral']) for d in matriz.loc[teste, 'dimensao']]) \
    * np.maximum(X_teste['media_7'].to_numpy(), 1.0)
cobertura = float(np.mean(np.abs(y_teste.to_numpy() - previsto_teste) <= margem_teste))
print(f'Cobertura alvo: {COBERTURA_ALVO:.0%} | cobertura medida no teste: {cobertura:.1%}')

## 7. Avaliação walk-forward (12 origens semanais)

Uma janela de teste só mede um recorte (e a última semana do extrato cai no Natal). O walk-forward
retreina a configuração vencedora a cada origem, só com o passado dela, e mede a semana seguinte.

In [ ]:
def particoes_walk_forward(matriz, janelas=12, passo=7, coluna='data_alvo'):
    fim = pd.Timestamp(matriz[coluna].max())
    for corte in [fim - pd.Timedelta(days=passo * i) for i in range(janelas, 0, -1)]:
        mascara_treino = matriz[coluna] < corte
        mascara_teste = (matriz[coluna] >= corte) & (matriz[coluna] < corte + pd.Timedelta(days=passo))
        if mascara_treino.sum() and mascara_teste.sum():
            yield corte, mascara_treino, mascara_teste


resultados = []
for corte, mascara_treino, mascara_teste in particoes_walk_forward(matriz):
    modelo_janela = instanciar(melhor['definicao'], melhor['parametros'])
    modelo_janela.fit(X[mascara_treino], y[mascara_treino])
    previsto = np.clip(modelo_janela.predict(X[mascara_teste]), 0, None)
    r = metricas_regressao(y[mascara_teste], previsto, X.loc[mascara_teste, 'mesmo_dow_1'].to_numpy())
    resultados.append({'origem': corte.date(), 'mae': r['mae'], 'mae_baseline': r['mae_baseline'],
                       'mase': r['mase'], 'n': r['n']})

walk_forward = pd.DataFrame(resultados)
print(f"MAE médio: {walk_forward['mae'].mean():.3f} | MASE médio: {walk_forward['mase'].mean():.3f} | "
      f"vitórias sobre a baseline: {(walk_forward['mae'] < walk_forward['mae_baseline']).sum()} de {len(walk_forward)}")
walk_forward.round(3)

## 8. Explicabilidade

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance


def estimador_final(modelo):
    """Desembrulha Pipeline e calibrador até o estimador que carrega os pesos."""
    if isinstance(modelo, Pipeline):
        return estimador_final(modelo.steps[-1][1])
    if type(modelo).__name__ in ('CalibratedClassifierCV', 'FrozenEstimator'):
        interno = getattr(modelo, 'estimator', None)
        if interno is not None:
            return estimador_final(interno)
    return modelo


def importancia_por_permutacao(modelo, X, y, scoring, repeticoes=5, limite=20):
    """Quanto a métrica piora ao embaralhar cada feature — medido na validação, fora do treino."""
    resultado = permutation_importance(modelo, X, y, scoring=scoring, n_repeats=repeticoes,
                                       random_state=SEMENTE, n_jobs=1)
    return (pd.DataFrame({'feature': X.columns, 'queda_metrica': resultado.importances_mean,
                          'desvio': resultado.importances_std})
            .sort_values('queda_metrica', ascending=False).head(limite).reset_index(drop=True))

In [ ]:
permutacao = importancia_por_permutacao(modelo, X_validacao, y_validacao, scoring='neg_mean_absolute_error')
permutacao.round(4)

In [ ]:
# SHAP de árvore: contribuição assinada de cada feature em cada previsão.
import shap

estimador = estimador_final(modelo)
if hasattr(estimador, 'feature_importances_'):
    explicador = shap.TreeExplainer(estimador)
    amostra = X_teste.sample(min(2000, len(X_teste)), random_state=SEMENTE)
    shap.summary_plot(explicador.shap_values(amostra), amostra, max_display=15, show=True)
else:
    explicador = None
    print('O vencedor desta execução não é um modelo de árvore; SHAP de árvore não se aplica.')

## 9. Artefato

In [ ]:
import joblib

joblib.dump({'modelo': modelo, 'colunas': list(X.columns), 'margens': margens, 'explicador': explicador,
             'vencedor': melhor['nome'], 'hiperparametros': melhor['parametros'], 'teste': resultado_teste},
            ARTEFATOS / 'model_d1.joblib', compress=3)
print('Artefato gravado em', ARTEFATOS / 'model_d1.joblib')